# Fine cross-resonance calibration

This hardware-backed example follows the staged CR calibration flow: prerequisite checks, amplitude scouting, bare-CR coarse calibration, repeated-gate fine calibration, rotary/local-frame tuning, and final validation. No calibration stage mutates the calibration note; only the explicit commit step does.

Choose device-specific sweep ranges conservatively. A proposed update outside a measured range is rejected rather than extrapolated.

In [ ]:
import numpy as np

import qubex as qx
from qubex.contrib import (
    CrCalibrationOptions,
    CrCalibrationTolerances,
    CrGateParameters,
    build_ecr_gate,
    calibrate_bare_cr,
    calibrate_ecr_angle,
    calibrate_ecr_local_z,
    calibrate_ecr_phase,
    calibrate_un_echoed_cancellation,
    check_cr_prerequisites,
    commit_cr_calibration,
    optimize_ecr_rotary,
    scout_cr_operating_points,
    validate_ecr_gate,
)

## 1. Create and connect the experiment

Complete the ordinary one-qubit, readout, and qutrit-classifier calibrations before starting this notebook. The control X180 pulse is part of every echoed CR gate.

In [ ]:
exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
)
exp.connect()

cr_control = "Q00"
cr_target = "Q01"
cr_label = f"{cr_control}-{cr_target}"

options = CrCalibrationOptions(
    repetition_counts=tuple(range(9)),
    max_iterations=3,
    plot=True,
)
tolerances = CrCalibrationTolerances()

## 2. Check prerequisites and scout CR amplitude

The scouting score uses the bare ZX rate and transverse coherent errors. Inspect leakage and driven-coherence data separately before accepting an operating point.

In [ ]:
prerequisites = check_cr_prerequisites(exp, cr_control, cr_target)
if not prerequisites["passed"]:
    raise RuntimeError(prerequisites["reason"])

ramptime = 32.0
time_range = np.arange(0.0, 1024.0 + 1, 32.0)
initial_params = CrGateParameters(
    cr_amplitude=0.2,
    cr_phase=0.0,
    cr_lobe_duration=256.0,
    cr_ramptime=ramptime,
)
scout = scout_cr_operating_points(
    exp,
    cr_control,
    cr_target,
    cr_amplitudes=np.linspace(0.1, 0.5, 5),
    time_range=time_range,
    initial_params=initial_params,
    options=options,
)
if not scout["converged"]:
    raise RuntimeError(scout["reason"])
params = scout["proposed_params"]
scout["candidates"]

## 3. Coarse bare-CR calibration

This stage iterates the CR phase and target cancellation IQ while echo, rotary, detuning, and local-frame corrections are disabled. It replaces the coarse role of `obtain_cr_params()` without storing intermediate values.

In [ ]:
coarse = calibrate_bare_cr(
    exp,
    cr_control,
    cr_target,
    initial_params=params,
    time_range=time_range,
    tolerances=tolerances,
    options=options,
)
if not coarse["verified"]:
    raise RuntimeError(coarse["reason"])
params = coarse["proposed_params"]
build_ecr_gate(exp, cr_control, cr_target, params=params).plot()

## 4. Fine repeated-gate calibration

Use the order ZY → ZX → IX → IY → ZX. Phase sweeps include zero, amplitude sweeps include one, and cancellation sweeps include zero so every update is compared with the exact input candidate.

In [ ]:
phase = calibrate_ecr_phase(
    exp,
    cr_control,
    cr_target,
    initial_params=params,
    phase_offsets=np.linspace(-0.12, 0.12, 7),
    tolerances=tolerances,
    options=options,
)
if phase["verified"]:
    params = phase["proposed_params"]

angle = calibrate_ecr_angle(
    exp,
    cr_control,
    cr_target,
    initial_params=params,
    amplitude_scales=np.linspace(0.9, 1.1, 5),
    tolerances=tolerances,
    options=options,
)
if angle["verified"]:
    params = angle["proposed_params"]

cancellation = calibrate_un_echoed_cancellation(
    exp,
    cr_control,
    cr_target,
    initial_params=params,
    cancel_x_offsets=np.linspace(-0.04, 0.04, 5),
    cancel_y_offsets=np.linspace(-0.04, 0.04, 5),
    tolerances=tolerances,
    options=options,
)
if cancellation["verified"]:
    params = cancellation["proposed_params"]

angle_again = calibrate_ecr_angle(
    exp,
    cr_control,
    cr_target,
    initial_params=params,
    amplitude_scales=np.linspace(0.95, 1.05, 5),
    tolerances=tolerances,
    options=options,
)
if angle_again["verified"]:
    params = angle_again["proposed_params"]

## 5. Rotary and local-frame refinement

The rotary helper currently provides a repeated-ECR error-amplification baseline and explicitly reports dedicated HEAT as not run. Target local-Z can be inferred from target tomography; control local-Z still requires an independent control Ramsey or tomography measurement.

In [ ]:
rotary = optimize_ecr_rotary(
    exp,
    cr_control,
    cr_target,
    initial_params=params,
    rotary_x_values=np.linspace(-0.05, 0.05, 9),
    rotary_y_values=np.linspace(-0.02, 0.02, 5),
    tolerances=tolerances,
    options=options,
)
if rotary["verified"]:
    params = rotary["proposed_params"]

local_z = calibrate_ecr_local_z(
    exp,
    cr_control,
    cr_target,
    initial_params=params,
    tolerances=tolerances,
    options=options,
)
if local_z["verified"]:
    params = local_z["proposed_params"]
local_z["components"]

## 6. Validate and commit

Run an independent control-phase check before setting `control_local_z_checked=True`. Validation uses the same explicit candidate for repeated tomography, Bell tomography, reference RB, and interleaved RB. Committing updates only the in-memory calibration note by default; save it explicitly after inspecting the stored values.

In [ ]:
control_local_z_checked = (
    False  # Set True only after an independent control Ramsey/tomography check.
)
validation = validate_ecr_gate(
    exp,
    cr_control,
    cr_target,
    params=params,
    tolerances=tolerances,
    options=options,
    run_leakage=True,
    run_bell_tomography=True,
    run_irb=True,
    waive_control_local_z=control_local_z_checked,
)
validation["criteria"]

In [ ]:
if not validation["committable"]:
    raise RuntimeError(validation["reason"])

commit = commit_cr_calibration(exp, validation)
commit["stored_params"]

# Persist only after reviewing the in-memory entry above.
exp.calib_note.save()